In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from tqdm import tqdm
import json
import numpy as np
import os
import pickle
from utils import construct_qiskit_hamiltonian, construct_qiskit_hamiltonian
from datasets import concatenate_datasets
from scipy.sparse.linalg import eigsh

# Configuration
CHECKPOINT_DIR = "checkpoints"
BATCH_SIZE = 100
DATASET_NAME = "linuzj/graph-data-quantum"
OUTPUT_NAME = "valterUo/graph-data-quantum-updated"


def ensure_checkpoint_dir():
    """Create checkpoint directory if it doesn't exist"""
    if not os.path.exists(CHECKPOINT_DIR):
        os.makedirs(CHECKPOINT_DIR)


def save_checkpoint(data, step_name, batch_idx=None):
    """Save checkpoint data"""
    ensure_checkpoint_dir()
    if batch_idx is not None:
        filename = f"{CHECKPOINT_DIR}/{step_name}_batch_{batch_idx}.pkl"
    else:
        filename = f"{CHECKPOINT_DIR}/{step_name}_complete.pkl"
    
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f"Checkpoint saved: {filename}")


def load_checkpoint(step_name, batch_idx=None):
    """Load checkpoint data"""
    if batch_idx is not None:
        filename = f"{CHECKPOINT_DIR}/{step_name}_batch_{batch_idx}.pkl"
    else:
        filename = f"{CHECKPOINT_DIR}/{step_name}_complete.pkl"
    
    if os.path.exists(filename):
        with open(filename, 'rb') as f:
            return pickle.load(f)
    return None

c:\Users\valte\OneDrive - University of Helsinki\Desktop\rl-quantum\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

def process_eigenvalues_with_checkpoints(ds):
    """Process eigenvalues with checkpointing for all splits"""
    print("Adding largest eigenvalues with checkpointing...")
    
    # Check if complete step exists
    completed_ds = load_checkpoint("eigenvalues")
    if completed_ds is not None:
        print("Found completed eigenvalues checkpoint, skipping...")
        return completed_ds
    
    result_splits = {}
    
    # Process each split (train, test, etc.)
    for split_name in ds.keys():
        print(f"Processing {split_name} split for eigenvalues...")
        
        processed_batches = []
        total_batches = len(ds[split_name]) // BATCH_SIZE + (1 if len(ds[split_name]) % BATCH_SIZE > 0 else 0)
        
        for batch_idx in range(total_batches):
            # Check if this batch was already processed
            batch_result = load_checkpoint(f"eigenvalues_{split_name}", batch_idx)
            if batch_result is not None:
                print(f"Loading eigenvalues {split_name} batch {batch_idx} from checkpoint...")
                processed_batches.append(batch_result)
                continue
            
            # Process this batch
            start_idx = batch_idx * BATCH_SIZE
            end_idx = min((batch_idx + 1) * BATCH_SIZE, len(ds[split_name]))
            batch_data = ds[split_name].select(range(start_idx, end_idx))
            
            print(f"Processing eigenvalues {split_name} batch {batch_idx}/{total_batches-1}...")
            
            updated_exact_solutions = []
            for i in range(len(batch_data)):
                hamiltonian_expr = batch_data[i]["cost_hamiltonian"]
                exact_solution_str = batch_data[i]["exact_solution"]
                
                # Parse existing exact solution
                exact_solution = json.loads(exact_solution_str)
                
                # Compute largest eigenvalue
                cost_hamiltonian = construct_qiskit_hamiltonian(hamiltonian_expr)
                # Try to compute eigenvalues efficiently, fallback to approximation if needed
                # For small matrices, use exact computation
                
                try:
                    matrix = cost_hamiltonian.to_matrix()
                    eigenvalues = np.linalg.eigvalsh(matrix)
                except Exception as e:
                    try:
                        # For larger matrices, use sparse eigenvalue computation
                        sparse_matrix = cost_hamiltonian.to_matrix(sparse=True)
                        eigenvalues = eigsh(sparse_matrix, k=1, which='LR', return_eigenvectors=False)
                    except Exception as e2:
                        print(f"Warning: Failed to compute eigenvalues for sample {i}: {e2}")
                        # Fallback: estimate from Hamiltonian coefficients
                        eigenvalues = np.array([np.sum(np.abs(cost_hamiltonian.coeffs.real))])
                
                largest_eigenvalue = float(np.max(eigenvalues))
                
                # Add largest eigenvalue to exact solution
                exact_solution["largest_eigenvalue"] = largest_eigenvalue
                updated_exact_solutions.append(json.dumps(exact_solution))
            
            # Create updated batch
            batch_dict = {key: batch_data[key] for key in batch_data.features}
            batch_dict["exact_solution"] = updated_exact_solutions
            batch_result = Dataset.from_dict(batch_dict)
            
            # Save checkpoint for this batch
            save_checkpoint(batch_result, f"eigenvalues_{split_name}", batch_idx)
            processed_batches.append(batch_result)
        
        # Combine all processed batches for this split
        if processed_batches:
            result_splits[split_name] = concatenate_datasets(processed_batches)
        else:
            result_splits[split_name] = ds[split_name]
    
    # Create final dataset
    result_ds = DatasetDict(result_splits)
    
    # Save complete checkpoint
    save_checkpoint(result_ds, "eigenvalues")
    return result_ds

In [3]:
def process_hamiltonian_representation_with_checkpoints(ds):
    """Process Hamiltonian representation with checkpointing for all splits"""
    print("Adding Qiskit Hamiltonian representation with checkpointing...")
    
    # Check if complete step exists
    completed_ds = load_checkpoint("hamiltonian")
    if completed_ds is not None:
        print("Found completed Hamiltonian checkpoint, skipping...")
        return completed_ds
    
    result_splits = {}
    
    # Process each split (train, test, etc.)
    for split_name in ds.keys():
        print(f"Processing {split_name} split for Hamiltonian representation...")
        
        processed_batches = []
        total_batches = len(ds[split_name]) // BATCH_SIZE + (1 if len(ds[split_name]) % BATCH_SIZE > 0 else 0)
        
        for batch_idx in range(total_batches):
            # Check if this batch was already processed
            batch_result = load_checkpoint(f"hamiltonian_{split_name}", batch_idx)
            if batch_result is not None:
                print(f"Loading Hamiltonian {split_name} batch {batch_idx} from checkpoint...")
                processed_batches.append(batch_result)
                continue
            
            # Process this batch
            start_idx = batch_idx * BATCH_SIZE
            end_idx = min((batch_idx + 1) * BATCH_SIZE, len(ds[split_name]))
            batch_data = ds[split_name].select(range(start_idx, end_idx))
            
            try:
                print(f"Processing Hamiltonian {split_name} batch {batch_idx}/{total_batches-1}...")
                
                pauli_strings_list = []
                coefficients_list = []
                
                for i in range(len(batch_data)):
                    hamiltonian_expr = batch_data[i]["cost_hamiltonian"]
                    
                    # Get Qiskit representation
                    qiskit_hamiltonian = construct_qiskit_hamiltonian_old(hamiltonian_expr)
                    
                    # Extract Pauli strings and coefficients
                    pauli_strings = [str(pauli) for pauli in qiskit_hamiltonian.paulis]
                    coefficients = qiskit_hamiltonian.coeffs.real.tolist()
                    
                    pauli_strings_list.append(pauli_strings)
                    coefficients_list.append(coefficients)
                
                # Create updated batch
                batch_dict = {key: batch_data[key] for key in batch_data.features}
                batch_dict["qiskit_pauli_strings"] = pauli_strings_list
                batch_dict["qiskit_coefficients"] = coefficients_list
                batch_result = Dataset.from_dict(batch_dict)
                
                # Save checkpoint for this batch
                save_checkpoint(batch_result, f"hamiltonian_{split_name}", batch_idx)
                processed_batches.append(batch_result)
                
            except Exception as e:
                print(f"Error processing Hamiltonian {split_name} batch {batch_idx}: {e}")
                print("Saving progress and exiting...")
                break
        
        # Combine all processed batches for this split
        if processed_batches:
            from datasets import concatenate_datasets
            result_splits[split_name] = concatenate_datasets(processed_batches)
        else:
            result_splits[split_name] = ds[split_name]
    
    # Create final dataset
    result_ds = DatasetDict(result_splits)
    
    # Save complete checkpoint
    save_checkpoint(result_ds, "hamiltonian")
    return result_ds

In [ ]:
# Main processing
#try:
# Load original dataset
print("Loading original dataset...")
ds = load_dataset(DATASET_NAME)

# Step 1: Add largest eigenvalues with checkpointing
ds_with_eigenvalues = process_eigenvalues_with_checkpoints(ds)

# Step 2: Add Qiskit Hamiltonian representation with checkpointing
extended_ds = process_hamiltonian_representation_with_checkpoints(ds_with_eigenvalues)

# Push to hub
print("Pushing to Hugging Face Hub...")
extended_ds.push_to_hub(OUTPUT_NAME)
print("Successfully uploaded to Hub!")

# Clean up checkpoints after successful completion
import shutil
if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
    print("Cleaned up checkpoint files")

#except Exception as e:
#    print(f"Process failed with error: {e}")
#    print("Checkpoints are saved. You can resume by running the script again.")

Loading original dataset...
Adding largest eigenvalues with checkpointing...
Processing train split for eigenvalues...
Processing eigenvalues train batch 0/139...
